In [1]:
import os
import glob
import pandas as pd

def combine_sensorData(file1, file2, fileProcessed):
    f1 = pd.read_csv(file1)
    f2 = pd.read_csv(file2)
    
    df = pd.merge_asof(f1, f2, on="Time (s)")
        
    df.to_csv(fileProcessed)
    return df

def downsample_class(df):
    # Downsample each recording separately
    df = df.iloc[::4] # Take every 4th sample
    return df.reset_index(drop=True)

root_folder = "C:\\Users\\JOSS0001\\OneDrive - Högskolan Kristianstad\\CTerminator 2\\Lab2_prep_JovanSsettuba"

all_data = []

for root, folders, files in os.walk(root_folder):
    # Check if this folder contains both sensor files
    if "Accelerometer.csv" in files and "Gyroscope.csv" in files:

        # Get the label from the parent folder
        # Example: ...\jumping\jumping1 2026-09-15 17-52-12
        label = os.path.basename(os.path.dirname(root))

        # Get recording folder name
        # Example: jumping1 2026-09-15 17-52-12
        recording_folder = os.path.basename(root)

        # Get recording ID from the beginning
        # Example: jumping1
        recording_id = recording_folder.split(" ")[0]

        # File paths
        accelerometer_path = os.path.join(root, "Accelerometer.csv")
        gyro_path = os.path.join(root, "Gyroscope.csv")

        # Read files
        acc = pd.read_csv(accelerometer_path)
        gyro = pd.read_csv(gyro_path)

        # Merge on time
        df = pd.merge(
            acc,
            gyro,
            on="Time (s)",
            how="inner"
        )

        # Add label and recording ID
        df.columns = ["time", "ax", "ay", "az", "gx", "gy", "gz"]
        df["label"] = label
        df["recording_id"] = recording_id

        #check for class to downsample
        if label == "jumping" or label == "skipping": 
            df = downsample_class(df)

        # Store result
        all_data.append(df)

# Combine all recordings
final_df = pd.concat(all_data, ignore_index=True)
final_df.to_csv("FinalDataFrame5.csv")


In [29]:
import matplotlib.pyplot as plt

def plot_recordings(folder, classname):
    count = 1

    while (count <= 12):
        file = f"C:\\Users\\JOSS0001\\OneDrive - Högskolan Kristianstad\\CTerminator 2\\actual_Lab2_JovanSsettuba\\{folder}\\{classname}{count}.csv"
        df = pd.read_csv(file, index_col=0)

        plt.figure()
        plt.plot(df["ax"], label="X")
        plt.plot(df["ay"], label="Y")
        plt.plot(df["az"], label="Z")

        
        plt.xlabel("Sample")
        plt.ylabel("Gyroscope")
        plt.title(f"{classname}{count}")
        plt.legend()
        
        count += 4
    
#plot_recordings("jump", "jumping")
#plot_recordings("run", "running")
# plot_recordings("skip", "skipping")
#plot_recordings("walk", "walking")

In [33]:
# Split the data into training and test sets
import numpy as np

np.random.seed(42)

df = pd.read_csv("FinalDataFrame.csv", index_col=0)

test_ids = []
train_ids = []

for label in df["label"].unique(): #Gets all labels in df
    ids = df[df["label"] == label]["recording_id"].unique() #All recs for each label

    #Pick 2 recs for test
    test_selected = np.random.choice(ids, size=2, replace=False)
    test_ids.extend(test_selected)

    #Remove test recs
    for i in ids:
        if i not in test_selected:
            train_ids.append(i)


training_df = df[df["recording_id"].isin(train_ids)]
test_df = df[df["recording_id"].isin(test_ids)]

#Save the separate sets
training_df.to_csv("training_set3.csv")
test_df.to_csv("test_set3.csv")

In [31]:
#*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=* 
# Combine ben and my data into one dataframe
#*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=* 

both = []
mine = pd.read_csv("test_set3.csv", index_col=0)
both.append(mine)

In [26]:
print(mine["label"].unique())

['jumping' 'running' 'skipping' 'walking']


In [28]:
benny = pd.read_csv("ben_test_set.csv", index_col=0)
#df
#df = pd.read_csv("training_set3.csv", index_col=0)
#df

benny["label"] = benny["label"].replace({
    "biking": "ben_bike",
    "running": "ben_run",
    "jumping": "ben_jump",
    "walking": "ben_walk"
})

both.append(benny)

In [20]:
print(benny["label"].unique())

['ben_bike' 'ben_jump' 'ben_run' 'ben_walk']


In [29]:
# Combine all recordings
both_df = pd.concat(both, ignore_index=True)
both_df.to_csv("Renamed_combined.csv")

In [30]:
benny = pd.read_csv("Renamed_combined.csv", index_col=0)
benny

,time,ax,ay,az,gx,gy,gz,label,recording_id
0,0.023974,-2.005050,-1.704000,-3.318000,-0.082775,0.142862,-0.537075,jumping,jumping7
1,0.040498,-6.006001,-0.483000,-5.146950,-1.708987,-0.640337,1.064662,jumping,jumping7
2,0.049941,-6.748050,-0.138000,-3.531000,-2.687575,-0.855250,2.625975,jumping,jumping7
3,0.059383,-4.342050,0.402000,1.120050,-3.513400,-0.674438,4.267862,jumping,jumping7
4,0.068825,0.466950,-0.700950,6.105000,-3.677162,0.095150,5.212350,jumping,jumping7
...,...,...,...,...,...,...,...,...,...
33933,NaN,8.670393,-15.148661,-0.845564,-1.117240,0.204634,0.360400,ben_walk,walking12
33934,NaN,10.609476,-15.153447,-0.146560,-1.618287,-0.444239,0.024434,ben_walk,walking12
33935,NaN,6.295217,-15.385850,0.325125,-2.081309,-1.271326,-0.544876,ben_walk,walking12
33936,NaN,-1.794318,-16.975283,-0.495912,-2.609081,-1.615844,-1.060431,ben_walk,walking12
